# PASSO 2: Notebook de cálculo de materiais, CO2 e custos

## Documentação: função processar_analise_consolidada

**Resumo:**

A função `processar_analise_consolidada(path_consolidado, path_fatores_emissao)` lê os dados consolidados de materiais e ambientes (consolidado_geral.xlsx), aplica regras para adicionar aço e lajotas, calcula emissões de CO2 (mín/máx), expande insumos-pai em componentes para cálculo detalhado de custo e gera um resumo normalizado por pavimento (kg/m² e R$/m²). Em seguida grava três novas abas no arquivo de entrada: `QUANTITATIVOS`, `CUSTO_CALCULADO` e `Consolidado_Pavimento`.

### Requisitos:
- Entradas:
  - `path_consolidado` (str): Excel com abas `Materiais_Padronizados` e `Ambientes_Padronizados` (consolidado_geral.xlsx).
  - `path_fatores_emissao` (str): arquivo com fatores de emissão e custos. Colunas esperadas como `insumo`, `fator_emissao_min`, `fator_emissao_max`, `custo_medio` (DIM_FATORES_EMISSAO.xlsx).
- Saídas:
  - Atualiza `path_consolidado` com as abas: `QUANTITATIVOS`, `CUSTO_CALCULADO`, `Consolidado_Pavimento`.
  - Efeitos colaterais: grava arquivos e imprime logs/avisos.

### Principais passos internos:
1. Lê arquivos de entrada (materiais, ambientes, fatores).
2. Executa `_add_steel_and_tiles` para inserir aço e ajustar lajotas.
3. Calcula `co2_min` e `co2_max` por linha com `_calcular_co2`.
4. Expande insumos-pai em componentes e calcula custos com `_calcular_custo`.
5. Gera resumo por pavimento (métricas por m²) com `_gerar_consolidado_pavimento`.
6. Salva as três abas no Excel de saída (modo append/replace de abas).

### Pré-condições / colunas obrigatórias:
- `Materiais_Padronizados`: colunas mínimas: `insumo`, `unidade_padrao`, `id_moradia`, `pavimento_bim`, `elemento` (além de `area_bim`/`volume_bim` quando aplicável).
- `Ambientes_Padronizados`: colunas mínimas: `id_moradia`, `pavimento_bim`, `area_bim`.
- `path_fatores_emissao`: colunas com `insumo`, `fator_emissao_min`, `fator_emissao_max`, `custo_medio` (nomes conforme uso no notebook).

### Pontos sensíveis e recomendações:
- Fechar o arquivo Excel (`path_consolidado`) antes de executar para evitar erros de escrita.
- Validar nomes exatos dos insumos-pai (ex.: "Concreto São Remo") se estiver usando expansão de componentes.
- Garantir conversão numérica: o código usa `pd.to_numeric(..., errors='coerce')` e trata NaNs quando necessário.
- Resetar índice antes de operações com máscaras para evitar ValueError em atribuições por `.loc` (já aplicado em pontos críticos).

### Uso rápido (exemplo):
```python
PATH_ARQUIVO_PRINCIPAL = r"/Users/.../consolidado_geral.xlsx"
PATH_FATORES_EMISSAO = r"/Users/.../DIM_FATORES_EMISSAO.xlsx"
processar_analise_consolidada(path_consolidado=PATH_ARQUIVO_PRINCIPAL,
                             path_fatores_emissao=PATH_FATORES_EMISSAO)
```

### Campos customizáveis:
- O corpo da função `processar_analise_consolidada` inicia com um bloco de configuração, onde a composição de concretos e argamassas podem ser ajustados de acordo com a convenção adotada para a cultura local.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Callable, Optional, Dict, Any

# ==============================================================================
# 1. LÓGICA DE ADIÇÃO DE AÇO E LAJOTAS (HELPER INTERNO)
# ==============================================================================
def _add_steel_and_tiles(
    df: pd.DataFrame,
    # --- aço ---
    steel_factor: float = 50.0, # 50 kg de aço por m³ de concreto
    steel_insumo_name: str = "Vergalhão de Aço CA-50",
    steel_condition: Optional[Callable[[pd.DataFrame], pd.Series]] = None,
    # --- lajotas ---
    tiles_factor: float = 13.0, # 13 unidades por m²
    tiles_condition: Optional[Callable[[pd.DataFrame], pd.Series]] = None,
    # --- colunas ---
    insumo_col: str = "insumo",
    qty_col: str = "unidade_padrao",
    area_col: str = "area_bim",
    vol_col: str = "volume_bim",
) -> pd.DataFrame:
    """
    (HELPER) Adiciona aço (novas linhas) e lajotas (modifica linhas existentes).
    Remove a lógica de salvar arquivos intermediários.
    """
    for col in [insumo_col, qty_col]:
        if col not in df.columns:
            raise KeyError(f"Coluna obrigatória ausente: '{col}'")

    out = df.copy()
    out[qty_col] = pd.to_numeric(out[qty_col], errors="coerce")

    # -------- AÇO --------
    if steel_condition is None:
        steel_condition = lambda d: d[insumo_col].astype(str).str.fullmatch(
            r"(?i)\s*concreto\s+s[aã]o\s+remo\s*", na=False
        )
    mask_steel = steel_condition(out)

    if mask_steel.any():
        steel_rows = out.loc[mask_steel].copy()
        steel_rows[qty_col] = steel_rows[qty_col] * steel_factor
        steel_rows[insumo_col] = steel_insumo_name
        if area_col in steel_rows.columns:
            steel_rows[area_col] = np.nan
        if vol_col in steel_rows.columns:
            steel_rows[vol_col] = np.nan
        
        # Adiciona uma coluna de tag para rastreio
        steel_rows["origem_registro"] = "componente_aco"
        out = pd.concat([out, steel_rows], ignore_index=True)

    # -------- LAJOTAS --------
    if tiles_condition is None:
        tiles_condition = lambda d: d.get("insumo", pd.Series("", index=d.index)).astype(str).str.contains("Lajota Enchimento", case=False, na=False)
    mask_tiles = tiles_condition(out)

    out[qty_col] = np.where(mask_tiles, out[qty_col] * tiles_factor, out[qty_col])
    
    print("  -> Aço e lajotas processados.")
    return out

# ==============================================================================
# 2. LÓGICA DE CÁLCULO DE CO2 MÁXIMO E MÍNIMO (HELPER INTERNO)
# ==============================================================================
def _calcular_co2(
    df_quantitativos: pd.DataFrame,
    df_fe: pd.DataFrame
) -> pd.DataFrame:
    """
    (HELPER) Calcula CO2 min/max no DataFrame.
    """
    df = df_quantitativos.copy()
    
    # Renomeia colunas de fatores para o merge
    df_fe_co2 = df_fe.rename(columns={
        "fator_emissao_min": "fe_min",
        "fator_emissao_max": "fe_max",
        "unidade": "fe_unidade"
    })
    
    # Remove colunas de CO2 se já existirem (para evitar duplicatas no merge)
    cols_to_drop = ["fe_min", "fe_max", "fe_unidade", "co2_min", "co2_max"]
    for col in cols_to_drop:
        if col in df.columns:
            df = df.drop(columns=col)

    # Merge por insumo
    df = df.merge(
        df_fe_co2[["insumo", "fe_min", "fe_max", "fe_unidade"]],
        on="insumo",
        how="left"
    )

    # Cálculo direto
    df["co2_min"] = df["unidade_padrao"] * df["fe_min"]
    df["co2_max"] = df["unidade_padrao"] * df["fe_max"]
    
    print("  -> CO2 min/max calculado.")
    return df

# ==============================================================================
# 3. LÓGICA DE CÁLCULO DE CUSTO (HELPER INTERNO)
# ==============================================================================

def _expandir_componentes(
    df_quantitativos: pd.DataFrame,
    insumos_pai: set,
    composicao: dict
) -> pd.DataFrame:
    """(HELPER) Expande insumos 'pai' (Concreto, Argamassa) em componentes."""
    df = df_quantitativos.copy()

    mask_pai = df["insumo"].isin(insumos_pai)
    pais = df.loc[mask_pai].copy()
    if pais.empty:
        print("  -> Nenhum insumo 'pai' encontrado para expandir.")
        return df

    novas_linhas = []
    for _, row in pais.iterrows():
        insumo_pai = row["insumo"]
        qtd_pai    = pd.to_numeric(row["unidade_padrao"], errors="coerce")
        if pd.isna(qtd_pai) or qtd_pai == 0:
            continue

        comp_map = composicao.get(insumo_pai, {})
        for componente, coef in comp_map.items():
            if coef and coef != 0:
                nova = row.copy()
                nova["insumo"] = componente
                nova["unidade_padrao"] = qtd_pai * coef
                nova["origem_registro"] = f"componente_de_{insumo_pai}"
                # Limpa CO2 do pai para não ser contado em duplicidade
                nova["co2_min"] = np.nan
                nova["co2_max"] = np.nan
                novas_linhas.append(nova)

    if novas_linhas:
        df = pd.concat([df, pd.DataFrame(novas_linhas)], ignore_index=True)
        print(f"  -> {len(novas_linhas)} linhas de componentes (Cimento, etc.) adicionadas.")

    return df

def _calcular_custo(
    df_quant_co2: pd.DataFrame,
    df_fe: pd.DataFrame,
    insumos_pai: set,
    composicao: dict,
    insumos_com_custo: set
) -> pd.DataFrame:
    """(HELPER) Expande componentes e calcula o custo."""
    
    # 1. Expandir
    df_exp = _expandir_componentes(df_quant_co2, insumos_pai, composicao)

    # 2. Merge com custos
    df_fe_cost = df_fe[["insumo", "custo_medio"]].copy()
    df_fe_cost["custo_medio"] = pd.to_numeric(df_fe_cost["custo_medio"], errors="coerce")
    
    # Evitar colunas duplicadas
    if "custo_medio" in df_exp.columns:
        df_exp = df_exp.drop(columns="custo_medio")

    df_cost = df_exp.merge(df_fe_cost, on="insumo", how="left")

    # 3. Calcular custo por linha
    df_cost["unidade_padrao"] = pd.to_numeric(df_cost["unidade_padrao"], errors="coerce")
    df_cost["custo_total"] = df_cost["unidade_padrao"] * df_cost["custo_medio"]

    # 4. Filtrar
    tem_custo = df_cost["custo_total"].notna() & (df_cost["custo_total"] != 0)
    insumo_alvo = df_cost["insumo"].isin(insumos_com_custo)
    df_cost_ok = df_cost.loc[tem_custo & insumo_alvo].copy()

    # 5. (Opcional) Agrupar por elemento
    if not df_cost_ok.empty:
      df_cost_ok["custo_total_elemento"] = (
          df_cost_ok.groupby(["id_moradia", "pavimento_bim", "elemento"])["custo_total"]
                   .transform("sum")
      )
    
    print(f"  -> Custo total calculado para {len(df_cost_ok)} linhas de insumos.")
    return df_cost_ok

# ==============================================================================
# 4. LÓGICA DO CONSOLIDADO POR M² (HELPER)
# ==============================================================================
def _gerar_consolidado_pavimento(
    df_quantitativos: pd.DataFrame,
    df_custo: pd.DataFrame,
    df_ambientes: pd.DataFrame
) -> pd.DataFrame:
    """(HELPER) Gera a nova aba de consolidação por pavimento (R$/m², kg/m²)."""
    
    # 1. Calcular área total por pavimento
    df_ambientes["area_bim"] = pd.to_numeric(df_ambientes["area_bim"], errors="coerce").fillna(0)
    area_por_pavimento = (
        df_ambientes.groupby(["id_moradia", "pavimento_bim"])["area_bim"]
        .sum()
        .reset_index()
        .rename(columns={"area_bim": "area_total_pavimento"})
    )
    # Filtrar pavimentos sem área
    area_por_pavimento = area_por_pavimento[area_por_pavimento["area_total_pavimento"] > 0]
    
    if area_por_pavimento.empty:
        print("  -> AVISO: Nenhuma área de ambiente encontrada. Aba consolidada por m² ficará vazia.")
        return pd.DataFrame()

    # 2. Agregar métricas
    # CO2 (do df principal, já que inclui tudo)
    agg_co2 = (
        df_quantitativos.groupby(["id_moradia", "pavimento_bim"])[["co2_min", "co2_max"]]
        .sum(numeric_only=True)
        .reset_index()
    )
    
    # Custo (apenas do df de custo)
    agg_custo = (
        df_custo.groupby(["id_moradia", "pavimento_bim"])["custo_total"]
        .sum(numeric_only=True)
        .reset_index()
    )
    
    # Aço (do df principal, 'unidade_padrao' é kg)
    mask_aco = df_quantitativos["insumo"] == "Vergalhão de Aço CA-50"
    agg_aco = (
        df_quantitativos[mask_aco]
        .groupby(["id_moradia", "pavimento_bim"])["unidade_padrao"]
        .sum(numeric_only=True)
        .reset_index()
        .rename(columns={"unidade_padrao": "total_aco_kg"})
    )
    
    # Cimento (do df de custo, 'unidade_padrao' é kg)
    # Usamos df_custo pois ele contém os componentes expandidos
    mask_cimento = df_custo["insumo"] == "Cimento"
    agg_cimento = (
        df_custo[mask_cimento]
        .groupby(["id_moradia", "pavimento_bim"])["unidade_padrao"]
        .sum(numeric_only=True)
        .reset_index()
        .rename(columns={"unidade_padrao": "total_cimento_kg"})
    )

    # 3. Juntar tudo
    df_summary = area_por_pavimento.merge(agg_co2, on=["id_moradia", "pavimento_bim"], how="left")
    df_summary = df_summary.merge(agg_custo, on=["id_moradia", "pavimento_bim"], how="left")
    df_summary = df_summary.merge(agg_aco, on=["id_moradia", "pavimento_bim"], how="left")
    df_summary = df_summary.merge(agg_cimento, on=["id_moradia", "pavimento_bim"], how="left")

    # Preencher com 0 onde não houver correspondência
    cols_agg = ["co2_min", "co2_max", "custo_total", "total_aco_kg", "total_cimento_kg"]
    df_summary[cols_agg] = df_summary[cols_agg].fillna(0)

    # 4. Normalizar (Calcular por m²)
    area = df_summary["area_total_pavimento"]
    df_summary["CO2 mínimo (kg/m2)"] = df_summary["co2_min"] / area
    df_summary["CO2 máximo (kg/m2)"] = df_summary["co2_max"] / area
    df_summary["Aço (kg/m2)"] = df_summary["total_aco_kg"] / area
    df_summary["Cimento (kg/m2)"] = df_summary["total_cimento_kg"] / area
    df_summary["Custo médio total (R$/m2)"] = df_summary["custo_total"] / area
    
    # Lidar com divisão por zero (embora já filtrado, é uma boa prática)
    df_summary = df_summary.replace([np.inf, -np.inf], np.nan)
    
    print("  -> Aba 'Consolidado_Pavimento' (por m²) gerada.")
    return df_summary


# ==============================================================================
# 5. FUNÇÃO PRINCIPAL UNIFICADA
# ==============================================================================
def processar_analise_consolidada(
    path_consolidado: str,
    path_fatores_emissao: str
):
    """
    Função unificada que lê o arquivo consolidado, aplica as lógicas
    de Aço/Lajota, CO2, Custo e gera a análise consolidada por m².
    
    Salva 3 novas abas no mesmo arquivo:
    - QUANTITATIVOS (Materiais + Aço/Lajota + CO2)
    - CUSTO_CALCULADO (Detalhamento de Custo com componentes)
    - Consolidado_Pavimento (Métricas por m² por pavimento)
    """
    
    # --- CONFIGURAÇÕES (do script original) ---
    INSUMOS_PAI = {
        "Concreto São Remo",
        "Concreto Contrapiso São Remo",
        "Argamassa São Remo",
    }
    COMPOSICAO = {
        "Concreto São Remo": {
            "Brita": 1037.28, # kg por m³ de concreto
            "Areia": 704.04, # kg por m³ de concreto
            "Cimento": 338.98, # kg por m³ de concreto
        },
        "Concreto Contrapiso São Remo": {
            "Brita": 842.79, # kg por m³ de concreto
            "Areia": 953.38, # kg por m³ de concreto
            "Cimento": 275.42, # kg por m³ de concreto
        },
        "Argamassa São Remo": {
            "Areia": 1315.47, # kg por m³ de argamassa
            "Brita": 0.00, # kg por m³ de argamassa
            "Cimento": 316.69, # kg por m³ de argamassa
        },
    }
    INSUMOS_COM_CUSTO = {
        "Revestimento cerâmico", "Telha de fibrocimento", "Vergalhão de Aço CA-50",
        "Bloco cerâmico", "Lajota Enchimento", "Bloco de concreto",
        "Brita", "Areia", "Cimento",
    }
    # --- Fim das Configurações ---

    print(f"Iniciando análise consolidada do arquivo: {path_consolidado}")
    
    try:
        # --- ETAPA 0: LEITURA ---
        print("Etapa 0: Lendo arquivos de entrada...")
        df_materiais = pd.read_excel(path_consolidado, sheet_name="Materiais_Padronizados")
        df_ambientes = pd.read_excel(path_consolidado, sheet_name="Ambientes_Padronizados")
        df_fe = pd.read_excel(path_fatores_emissao)
        print("  -> Arquivos lidos com sucesso.")

        # --- ETAPA 1: AÇO E LAJOTAS ---
        print("Etapa 1: Adicionando Aço e Lajotas...")
        df_quantitativos = _add_steel_and_tiles(df_materiais)

        # --- ETAPA 2: CÁLCULO DE CO2 ---
        print("Etapa 2: Calculando CO2...")
        df_quantitativos = _calcular_co2(df_quantitativos, df_fe)

        # --- ETAPA 3: CÁLCULO DE CUSTO ---
        print("Etapa 3: Calculando Custos (com expansão de componentes)...")
        df_custo = _calcular_custo(
            df_quantitativos, 
            df_fe, 
            INSUMOS_PAI, 
            COMPOSICAO, 
            INSUMOS_COM_CUSTO
        )

        # --- ETAPA 4: CONSOLIDADO POR PAVIMENTO (m²) ---
        print("Etapa 4: Gerando consolidado por pavimento...")
        df_summary = _gerar_consolidado_pavimento(
            df_quantitativos,
            df_custo,
            df_ambientes
        )

        # --- ETAPA 5: SALVAR RESULTADOS ---
        print(f"Etapa 5: Salvando resultados em {path_consolidado}...")
        # Usamos mode='a' (append) e if_sheet_exists='replace' para
        # adicionar/substituir apenas as abas de análise,
        # mantendo as abas originais intactas.
        with pd.ExcelWriter(
            path_consolidado, 
            mode="a", 
            engine="openpyxl", 
            if_sheet_exists="replace"
        ) as writer:
            
            # Aba com materiais + aço/lajota + CO2
            df_quantitativos.to_excel(writer, sheet_name="QUANTITATIVOS", index=False)
            
            # Aba com detalhamento de custos (inclui Cimento, Areia, etc.)
            df_custo.to_excel(writer, sheet_name="CUSTO_CALCULADO", index=False)
            
            # Nova aba de resumo por m²
            df_summary.to_excel(writer, sheet_name="Consolidado_Pavimento", index=False)

        print(f"✅ Processo concluído! Arquivo atualizado com 3 abas de análise.")

    except FileNotFoundError as e:
        print(f"ERRO: Arquivo não encontrado. Verifique os caminhos.")
        print(e)
    except ValueError as e:
        print(f"ERRO: Aba não encontrada. Verifique os nomes das abas ('Materiais_Padronizados', 'Ambientes_Padronizados').")
        print(e)
    except KeyError as e:
        print(f"ERRO: Coluna não encontrada. Verifique se os arquivos de entrada possuem as colunas esperadas (ex: 'insumo', 'unidade_padrao').")
        print(e)
    except Exception as e:
        print(f"Ocorreu um erro inesperado durante o processamento:")
        print(e)


# Aplicação da função de cálculo

In [2]:
# --- Defina os caminhos para seus arquivos ---
PATH_ARQUIVO_PRINCIPAL = r"/Users/camiladuelisviana/Desktop/MORE/Integracao/consolidado_geral.xlsx"
PATH_FATORES_EMISSAO = r"/Users/camiladuelisviana/Desktop/MORE/DIM_FATORES_EMISSAO.xlsx"

# --- Executar a função unificada ---
processar_analise_consolidada(
    path_consolidado=PATH_ARQUIVO_PRINCIPAL,
    path_fatores_emissao=PATH_FATORES_EMISSAO
)

Iniciando análise consolidada do arquivo: /Users/camiladuelisviana/Desktop/MORE/Integracao/consolidado_geral.xlsx
Etapa 0: Lendo arquivos de entrada...
  -> Arquivos lidos com sucesso.
Etapa 1: Adicionando Aço e Lajotas...
  -> Aço e lajotas processados.
Etapa 2: Calculando CO2...
  -> CO2 min/max calculado.
Etapa 3: Calculando Custos (com expansão de componentes)...
  -> 8577 linhas de componentes (Cimento, etc.) adicionadas.
  -> Custo total calculado para 11258 linhas de insumos.
Etapa 4: Gerando consolidado por pavimento...
  -> Aba 'Consolidado_Pavimento' (por m²) gerada.
Etapa 5: Salvando resultados em /Users/camiladuelisviana/Desktop/MORE/Integracao/consolidado_geral.xlsx...
✅ Processo concluído! Arquivo atualizado com 3 abas de análise.
